In [1]:
import numpy as np
from collections import defaultdict
from datetime import datetime
import string
file_name = "hostel_bois.txt"

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total Lines:", len(lines))
messages = []

participants = set()

system_messages = 0
media_messages = 0
deleted_messages = 0

media_per_person = defaultdict(int)
deleted_per_person = defaultdict(int)



Total Lines: 3178


In [2]:
for line in lines:

    line = line.strip()

    if line == "":
        continue

    # Skip continuation lines
    if len(line) < 8 or line[2] != "/" or line[5] != "/":
        continue

    try:

        timestamp, rest = line.split(" - ", 1)

    except ValueError:

        continue

    # System Message
    if ": " not in rest:
        system_messages += 1
        continue

    sender, message = rest.split(": ", 1)

    participants.add(sender)

    if message == "<Media omitted>":
        media_messages += 1
        media_per_person[sender] += 1

    if message == "This message was deleted":
        deleted_messages += 1
        deleted_per_person[sender] += 1

    messages.append({

        "timestamp": timestamp,
        "sender": sender,
        "message": message

    })
    print("="*60)
print("CHAT PARSER")
print("="*60)

print("Messages Parsed :", len(messages))
print("Participants :", len(participants))
print("System Messages :", system_messages)
print("Media Messages :", media_messages)
print("Deleted Messages :", deleted_messages)

print()

print("Participants")

for person in sorted(participants):
    print(person)
    for message in messages:

    message["datetime"] = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    message["date"] = message["datetime"].date()

    message["hour"] = message["datetime"].hour
    print()

print(messages[0])

print()

print(messages[1])

print()

print(messages[2])
# ===========================================
# FEATURE 2 - GROUP OVERVIEW
# ===========================================

message_count = defaultdict(int)

for msg in messages:
    message_count[msg["sender"]] += 1
    total_messages = len(messages)
    sorted_people = sorted(
    message_count.items(),
    key=lambda x: x[1],
    reverse=True
)
    first_date = min(msg["date"] for msg in messages)
last_date = max(msg["date"] for msg in messages)

total_days = (last_date - first_date).days + 1
print("=" * 65)
print("GROUP OVERVIEW")
print("=" * 65)

print(f"Group Name       : Hostel Bois 4ever")
print(f"Period           : {first_date} to {last_date}")
print(f"Total Days       : {total_days}")
print(f"Participants     : {len(participants)}")
print(f"Total Messages   : {total_messages}")
print()

print("-" * 65)
print("MESSAGES PER PERSON")
print("-" * 65)

for person, count in sorted_people:

    percentage = (count / total_messages) * 100

    print(f"{person:<10} {count:>5} messages ({percentage:.2f}%)")
    day_count = defaultdict(int)

for msg in messages:
    day_count[msg["date"]] += 1

busy_day = max(day_count, key=day_count.get)

print()
print("=" * 65)
print("MOST ACTIVE DAY")
print("=" * 65)

print(f"{busy_day} ({day_count[busy_day]} messages)")

hour_count = defaultdict(int)

for msg in messages:
    hour_count[msg["hour"]] += 1

busy_hour = max(hour_count, key=hour_count.get)

print()
print("=" * 65)
print("MESSAGES PER DAY")
print("=" * 65)

for day in sorted(day_count):
    print(day, ":", day_count[day])
    top_person = sorted_people[0]

print()
print("=" * 65)
print("MOST ACTIVE MEMBER")
print("=" * 65)

print("Name :", top_person[0])
print("Messages :", top_person[1])

percentage = (top_person[1] / total_messages) * 100

print(f"Contribution : {percentage:.2f}%")
least_person = sorted_people[-1]

print()
print("=" * 65)
print("LEAST ACTIVE MEMBER")
print("=" * 65)

print("Name :", least_person[0])
print("Messages :", least_person[1])

percentage = (least_person[1] / total_messages) * 100

print(f"Contribution : {percentage:.2f}%")
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)

print("✔ Total Participants :", len(participants))
print("✔ Total Messages :", total_messages)
print("✔ Total Days :", total_days)
print("✔ Most Active Member :", top_person[0])
print("✔ Least Active Member :", least_person[0])
print("✔ Busiest Day :", busy_day)
print(f"✔ Peak Hour : {busy_hour:02d}:00")
# ===========================================
# FEATURE 4 - ACTIVITY HEATMAP
# ===========================================

people = sorted(list(participants))

person_index = {}

for i, person in enumerate(people):
    person_index[person] = i
    heatmap = np.zeros((len(people), 24), dtype=int)
    for msg in messages:

    row = person_index[msg["sender"]]
    col = msg["hour"]

    heatmap[row][col] += 1
    print("=" * 90)
print("NUMPY HEATMAP (Message Count)")
print("=" * 90)

print("      ", end="")

for hour in range(24):
    print(f"{hour:>4}", end="")

print()

for i, person in enumerate(people):

    print(f"{person:<8}", end="")

    for value in heatmap[i]:
        print(f"{value:>4}", end="")

    print()
    print()
print("=" * 90)
print("ACTIVITY HEATMAP")
print("=" * 90)

print("Hour → ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()
for i, person in enumerate(people):

    print(f"{person:<8}", end=" ")

    maximum = max(heatmap[i])

    if maximum == 0:
        maximum = 1

    for value in heatmap[i]:

        ratio = value / maximum

        if ratio == 0:
            symbol = ". "

        elif ratio <= 0.25:
            symbol = "░ "

        elif ratio <= 0.50:
            symbol = "▒ "

        elif ratio <= 0.75:
            symbol = "▓ "

        else:
            symbol = "█ "

        print(symbol, end="")

    print()
    hour_totals = np.sum(heatmap, axis=0)

print()
print("=" * 60)
print("TOTAL GROUP ACTIVITY PER HOUR")
print("=" * 60)

for hour in range(24):
    print(f"{hour:02d}:00 -> {hour_totals[hour]}")
    person_totals = np.sum(heatmap, axis=1)

print()
print("=" * 60)
print("TOTAL MESSAGES (NumPy)")
print("=" * 60)

for i, person in enumerate(people):
    print(person, ":", person_totals[i])
    print()
print("=" * 60)
print("NIGHT ACTIVITY")
print("=" * 60)

for i, person in enumerate(people):

    night = (
        heatmap[i][23]
        + heatmap[i][0]
        + heatmap[i][1]
        + heatmap[i][2]
        + heatmap[i][3]
        + heatmap[i][4]
    )

    total = np.sum(heatmap[i])

    percentage = (night / total) * 100 if total else 0

    print(f"{person:<10} {percentage:.2f}%")
    from collections import defaultdict
    stop_words = {
    "i", "is", "the", "a", "an", "and", "or",
    "to", "of", "in", "on", "for", "at", "it",
    "this", "that", "you", "me", "my", "your",
    "we", "our", "are", "was", "be", "am",
    "with", "have", "has", "had", "but"
}
word_count = defaultdict(int)

for msg in messages:

    text = msg["message"]

    if text == "<Media omitted>":
        continue

    if text == "This message was deleted":
        continue

    words = text.lower().split()

    for word in words:

        word = word.strip(string.punctuation)

        if word == "":
            continue

        if word in stop_words:
            continue

        word_count[word] += 1
        top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("=" * 70)
print("TOP 10 WORDS")
print("=" * 70)

for word, count in top_words:
    print(f"{word:<20}{count}")
    print()

print("=" * 70)
print("GROUP FAVOURITE WORDS")
print("=" * 70)

highest = top_words[0][1]

for word, count in top_words:

    bar = "█" * int((count / highest) * 25)

    print(f"{word:<15}{bar} {count}")
    response_times = defaultdict(list)
    for i in range(1, len(messages)):

    previous = messages[i-1]
    current = messages[i]

    if previous["sender"] != current["sender"]:

        gap = (
            current["datetime"] -
            previous["datetime"]
        ).total_seconds()

        response_times[current["sender"]].append(gap)
        print()

print("=" * 70)
print("AVERAGE RESPONSE TIME")
print("=" * 70)

for person in people:

    gaps = response_times[person]

    if len(gaps) == 0:
        print(person, "No replies")
        continue

    average = sum(gaps) / len(gaps)

    minutes = average / 60

    print(f"{person:<10}{minutes:.2f} minutes")
    activity = defaultdict(set)

for msg in messages:
    activity[msg["sender"]].add(msg["date"])
    print()

print("=" * 70)
print("LONGEST SILENT STREAK")
print("=" * 70)

all_dates = sorted(day_count.keys())

for person in people:

    longest = 0
    current = 0

    for day in all_dates:

        if day in activity[person]:
            current = 0
        else:
            current += 1

            if current > longest:
                longest = current

    print(f"{person:<10}{longest} days")
    fastest = None
best = float("inf")

for person in people:

    gaps = response_times[person]

    if len(gaps) == 0:
        continue

    average = sum(gaps) / len(gaps)

    if average < best:
        best = average
        fastest = person

print()

print("=" * 70)
print("FASTEST REPLIER")
print("=" * 70)

print(fastest)
print(f"{best/60:.2f} minutes")
slowest = None
worst = 0

for person in people:

    gaps = response_times[person]

    if len(gaps) == 0:
        continue

    average = sum(gaps) / len(gaps)

    if average > worst:
        worst = average
        slowest = person

print()

print("=" * 70)
print("SLOWEST REPLIER")
print("=" * 70)

print(slowest)
print(f"{worst/60:.2f} minutes")
print("=" * 70)
print("PERSONALITY ARCHETYPES")
print("=" * 70)

archetypes = {}

for person in people:

    total = message_count[person]

    # Night messages (11 PM–4 AM)
    night = (
        heatmap[person_index[person]][23]
        + heatmap[person_index[person]][0]
        + heatmap[person_index[person]][1]
        + heatmap[person_index[person]][2]
        + heatmap[person_index[person]][3]
        + heatmap[person_index[person]][4]
    )

    night_percent = (night / total) * 100 if total else 0

    silent = 0

    current = 0

    for day in all_dates:

        if day in activity[person]:
            current = 0
        else:
            current += 1
            silent = max(silent, current)

    # Average message length
    lengths = []

    for msg in messages:

        if msg["sender"] == person:

            if msg["message"] in ["<Media omitted>", "This message was deleted"]:
                continue

            lengths.append(len(msg["message"]))

    average_length = sum(lengths) / len(lengths) if lengths else 0

    # Decide archetype

    if total == max(message_count.values()):
        archetype = "The Spammer"

    elif night_percent >= 70:
        archetype = "The Night Owl"

    elif silent >= 20:
        archetype = "The Ghost"

    elif average_length >= 120:
        archetype = "The Storyteller"

    elif media_per_person[person] >= 10:
        archetype = "The Meme Dealer"

    else:
        archetype = "The Regular"

    archetypes[person] = archetype

    print(f"{person:<10} → {archetype}")
    print("\n")
print("=" * 80)
print("             GROUPDNA FINAL REPORT")
print("=" * 80)

print(f"Group Name          : Hostel Bois 4ever")
print(f"Participants        : {len(participants)}")
print(f"Duration            : {total_days} days")
print(f"Messages            : {total_messages}")
print(f"System Messages     : {system_messages}")
print(f"Media Shared        : {media_messages}")
print(f"Deleted Messages    : {deleted_messages}")

print("\n")

print("-" * 80)
print("MOST ACTIVE MEMBER")
print("-" * 80)

print(f"{top_person[0]} ({top_person[1]} messages)")

print("\n")

print("-" * 80)
print("LEAST ACTIVE MEMBER")
print("-" * 80)

print(f"{least_person[0]} ({least_person[1]} messages)")

print("\n")

print("-" * 80)
print("BUSIEST DAY")
print("-" * 80)

print(f"{busy_day}")
print(f"{day_count[busy_day]} messages")

print("\n")

print("-" * 80)
print("PEAK HOUR")
print("-" * 80)

print(f"{busy_hour:02d}:00 - {busy_hour+1:02d}:00")

print("\n")

print("-" * 80)
print("FASTEST REPLIER")
print("-" * 80)

print(f"{fastest}")

print("\n")

print("-" * 80)
print("TOP WORDS")
print("-" * 80)

for word, count in top_words:
    print(f"{word:<15}{count}")

print("\n")

print("-" * 80)
print("PERSONALITY TYPES")
print("-" * 80)

for person in people:
    print(f"{person:<10}{archetypes[person]}")

print("\n")

print("=" * 80)
print("THANK YOU")
print("=" * 80)
print("\nMEDIA SHARES")

ranking = sorted(
    media_per_person.items(),
    key=lambda x: x[1],
    reverse=True
)

for person, count in ranking:
    print(person, count)
    print("\nDELETED MESSAGES")

ranking = sorted(
    deleted_per_person.items(),
    key=lambda x: x[1],
    reverse=True
)

for person, count in ranking:
    print(person, count)
    print("\nMOST ACTIVE HOUR")

for person in people:

    index = person_index[person]

    hour = np.argmax(heatmap[index])

    print(person, ":", f"{hour:02d}:00")
    with open("GroupDNA_Report.txt", "w", encoding="utf-8") as file:

    file.write("GROUPDNA REPORT\n")
    file.write("=" * 50 + "\n\n")

    file.write(f"Messages : {total_messages}\n")
    file.write(f"Participants : {len(participants)}\n")
    file.write(f"Duration : {total_days} days\n\n")

    file.write("Personality Archetypes\n")

    for person in people:
        file.write(f"{person} -> {archetypes[person]}\n")

print("Report saved successfully!")



IndentationError: expected an indented block after 'for' statement on line 60 (2593328445.py, line 62)